# Pretrain image segmentation model (unet) for feature generation

This notebook purpose is to pretrain a UNet model on the clean training split, on a classic binary segmentation task, in order to learn useful features for the upcoming model for path classification. The pretrained UNet will be used as a feature extractor for the path classification model, and will be fine-tuned on the path classification task in a second step.

For reproducibility, we provide this notebook to allow people to pretrain themselves the UNet model, but we also provide the pretrained model weights in the repository, so that it is not mandatory to run this notebook (see [README](../README.md)) to be able to train the path classification model.

In [ ]:
import os
from utils.available_datasets import available_datasets

dataset_choice = available_datasets["PERSEVERE"]

train_split = dataset_choice.preferred_train_split
data_dir = dataset_choice.data_dir
splits_filepath = dataset_choice.splits_filepath
ndim = dataset_choice.ndim
input_channels = dataset_choice.input_channels
model_config_filepath = dataset_choice.model_config_filepath
model_config = dataset_choice.model_config

In [ ]:
print(model_config)

## Data visualization
### Show dataset without data augmentation

In [ ]:
from image_segmentation.data.image_datamodule import ImageDatamodule
from image_segmentation.data.image_dataset import ImageDataset
from image_segmentation.data.augmentations import build_empty_transform

train_transforms = build_empty_transform(ndim)
val_transforms = build_empty_transform(ndim)

datamodule = ImageDatamodule(
    data_dir=data_dir,
    split_file_path=splits_filepath,
    train_split_name=train_split,
    val_split_ratio=0.2,
    train_transforms=train_transforms,
    val_transforms=val_transforms,
    test_transforms=val_transforms,
    num_workers=0,
    train_batch_size=1,
    val_batch_size=1,
    seed=42,
    shuffle_train=False
)

datamodule.setup()
dataloader = datamodule.train_dataloader()

In [ ]:
from image_segmentation.data.data_viz import plot_batch

for i, batch in enumerate(dataloader):
    plot_batch(batch, 
               ndim=ndim,
               plot_3d_mode="acc")
    del batch
    break

### Show dataset with data augmentation, crop and normalization

In [ ]:
stats = datamodule.dataset.get_dataset_stats(
    ndim=ndim,
    input_channels=input_channels,
    split_name=train_split,
    split_indices=datamodule.train_indices
)

In [ ]:
masked = "foreground"
if "foreground" not in stats.keys():
    masked = "full_image"
mean = stats[masked]["mean"]
std = stats[masked]["std"]
print(stats)

In [ ]:
from image_segmentation.data.augmentations import build_train_transform, build_val_transform

train_transforms = build_train_transform(ndim, mean, std, model_config['data']['train_patch_size'], fg_probability_3d=model_config['data']['train_fg_probability_3d'])
val_transforms = build_val_transform(ndim, mean, std)

datamodule = ImageDatamodule(
    data_dir=data_dir,
    split_file_path=splits_filepath,
    train_split_name=train_split,
    val_split_ratio=0.2,
    train_transforms=train_transforms,
    val_transforms=val_transforms,
    test_transforms=val_transforms,
    num_workers=0,
    train_batch_size=model_config['data']['train_batch_size'],
    val_batch_size=model_config['data']['val_batch_size'],
    seed=42,
    shuffle_train=False
)

datamodule.setup()

In [ ]:
from image_segmentation.data.data_viz import plot_batch

n_augmentations = 5

dataloader = datamodule.train_dataloader()
for j in range(n_augmentations):
    for i, batch in enumerate(dataloader):
        plot_batch(batch, ndim, plot_3d_mode="acc")
        del batch
        break

In [ ]:
dataloader = datamodule.val_dataloader()
for i, batch in enumerate(dataloader):
    plot_batch(batch, ndim, plot_3d_mode="acc")
    del batch
    break

### Initialize the BinarySegmentator model with the specified hyperparameters, ready for training on the preprocessed dataset.

In [ ]:
from image_segmentation.models import BinarySegmentator

binary_segmentator = BinarySegmentator(
    lr=1e-3,
    input_channels=input_channels,
    ndim=ndim,
    num_layers=model_config['unet']['num_layers'],
    features_start=model_config['unet']['features_start'],
    bilinear=True,
    norm_op=model_config['unet']['norm_op'],
    warmup_epochs=1,
    dropout=0.2,
    kernel_size=model_config['unet']['kernel_size'],
    dice_loss_ratio=model_config["training"]["dice_loss_ratio"],
    val_patch_size=model_config["data"]["val_patch_size"],
    val_patch_overlap=model_config["data"]["val_patch_overlap"],
    val_sw_batch_size=model_config["data"]["val_sw_batch_size"],
)

### Initialize the callbacks and pytorch-lightning trainer

In [ ]:
from pytorch_lightning import Trainer
import torch
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import CSVLogger

from image_segmentation.models.callbacks import SaveConfigCallback, PlotMetricsCallback

run_dir = dataset_choice.make_run_dir(base_dir="unet_pretraining", run_name=None)

logger = CSVLogger(save_dir=run_dir)

callbacks = [
    ModelCheckpoint(
        dirpath=run_dir,
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        filename="best-checkpoint-{epoch:02d}-{val_loss:.4f}"
    ),
    SaveConfigCallback(config=model_config, save_dir=run_dir),
    PlotMetricsCallback(save_dir=run_dir, metrics=["loss", "dice", "precision", "recall"], every_n_epochs=1),
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        min_delta=0.00,
        verbose=True,
        mode="min"
    )
]

trainer = Trainer(
    accelerator='auto', 
    devices="auto", 
    max_epochs=1000, 
    precision='16-mixed', 
    accumulate_grad_batches=model_config["training"]["accumulate_grad_batches"],
    callbacks=callbacks,
    logger=logger
)
torch.set_float32_matmul_precision("medium")

## Model training

In [ ]:
trainer.fit(binary_segmentator, datamodule=datamodule)

## Testing the model for binary segmentation task

In [ ]:
from utils.device import get_device

device = get_device()

ckpt_path = dataset_choice.get_checkpoint_by_id("unet_pretraining", -1)

model = BinarySegmentator.load_from_checkpoint(ckpt_path, map_location=device)

print(model)

In [ ]:
trainer.test(model, datamodule=datamodule)

In [ ]:
import matplotlib.pyplot as plt
import gc
import torch

from image_segmentation.data.data_viz import plot_batch

model = model.to(device)
model.eval()

with torch.no_grad():
    for i, batch in enumerate(datamodule.test_dataloader()):
        imgs, masks = batch
        imgs = imgs.to(device)
        masks = masks.to(device)

        logits = model._predict(imgs)
        binary_preds = (torch.sigmoid(logits) > 0.5).float()

        plot_batch(batch, ndim=ndim, pred=binary_preds.cpu(), plot_3d_mode="acc")
        
        del imgs, masks, logits, binary_preds
        if i >= 4:
            break

torch.cuda.empty_cache()
gc.collect()

Now that we have a trained model for feature extraction, you can continue on the [Segmentation predictions generation notebook (3)](./03_generate_segmentation_preds.ipynb)